# 09 Fine-tuning CamemBERT

Exploration du fine-tuning CamemBERT pour la classification de sentiment. **Résultat honnête** : le dataset d'entraînement (284 exemples) est insuffisant pour les classes minoritaires SOUTIEN et HOSTILITÉ — le modèle n'apprend pas à les prédire (F1 = 0). Ce notebook documente l'expérience et les pistes d'amélioration (augmentation de données, class weights, focal loss). La validation lexicale TF-IDF (CRITIQUE vs HOSTILITÉ = vocabulaires distincts) reste le point fort de la validation du pipeline.

## 9.1 Configuration

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import json
import warnings

warnings.filterwarnings("ignore")

_root = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(_root))
from src.utils import FIG_DIR, MODEL_DIR, DATA_RAW, load_replies

FIG_DIR.mkdir(parents=True, exist_ok=True)

## 9.2 Chargement du modèle

In [2]:
model_path = MODEL_DIR
if not model_path.exists() or not (model_path / "config.json").exists():
    raise FileNotFoundError(
        "Modèle non trouvé. Exécuter d'abord : python scripts/train_sentiment_bert.py"
    )

try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    import torch

    tokenizer = AutoTokenizer.from_pretrained(str(model_path))
    model = AutoModelForSequenceClassification.from_pretrained(str(model_path))
    model.eval()
    print("Modèle chargé : sentiment_camembert")
except ImportError:
    raise ImportError("pip install transformers torch")

Modèle chargé : sentiment_camembert


## 9.3 Prédictions sur un échantillon

In [3]:
replies = load_replies()
texts = replies["text"].dropna().astype(str)
n_sample = min(500, len(texts))
sample = texts.sample(n=n_sample, random_state=42)

batch_size = 32
predictions = []
with torch.no_grad():
    for i in range(0, len(sample), batch_size):
        batch = sample.iloc[i : i + batch_size].tolist()
        inp = tokenizer(batch, truncation=True, max_length=256, padding=True, return_tensors="pt")
        out = model(**inp.to(model.device))
        pred_ids = out.logits.argmax(dim=-1).cpu().numpy()
        predictions.extend(pred_ids.tolist())

id2label = model.config.id2label
pred_labels = [id2label.get(int(p), str(p)) for p in predictions]
sample_df = replies.loc[sample.index].copy()
sample_df["bert_pred"] = pred_labels

## 9.4 Comparaison BERT vs GPT-5 Nano

In [4]:
gpt_col = "sentiment"
if gpt_col in sample_df.columns:
    def norm(s):
        return s.str.upper().str.replace("É", "E").str.replace("HOSTILITÉ", "HOSTILITE")
    m = norm(sample_df["bert_pred"])
    g = norm(sample_df[gpt_col])
    agree = (m == g).sum()
    acc_agree = agree / len(sample_df)
    print(f"Accord BERT / GPT-5 Nano : {agree}/{len(sample_df)} ({acc_agree:.1%})")
    crosstab = pd.crosstab(sample_df["bert_pred"], sample_df[gpt_col], margins=True)
    print(crosstab.to_string())

Accord BERT / GPT-5 Nano : 127/500 (25.4%)
sentiment  CRITIQUE  HOSTILITE  INCONNU  IRONIE  SOUTIEN  All
bert_pred                                                    
CRITIQUE         90         58        3      13       53  217
IRONIE           79         76        2      37       89  283
All             169        134        5      50      142  500


## 9.5 Métriques officielles (test set)

In [5]:
metrics_path = Path("..").resolve() / "docs" / "bert_metrics.json"
if metrics_path.exists():
    with open(metrics_path, encoding="utf-8") as f:
        m = json.load(f)
    print("Accuracy (test) :", m.get("accuracy", "N/A"))
    print("F1 macro (test) :", m.get("f1_macro", "N/A"))
    print("F1 par classe :", m.get("f1_per_class", {}))
else:
    print("bert_metrics.json non trouvé (train_sentiment_bert.py non exécuté)")

Accuracy (test) : 0.18604651162790697
F1 macro (test) : 0.14451612903225808
F1 par classe : {'CRITIQUE': 0.32, 'HOSTILITE': 0.0, 'IRONIE': 0.25806451612903225, 'SOUTIEN': 0.0}


## 9.6 Post-mortem — pourquoi le fine-tuning échoue

| Hypothèse | Testée ? | Résultat | Effort restant |
|-----------|----------|----------|----------------|
| Dataset trop petit (284 exemples) | ✅ | Oui — 71/classe en moyenne, insuffisant pour 4 classes | Collecter 500+ annotations |
| Déséquilibre des classes | ✅ WeightedTrainer | Marginal — pas d'amélioration significative | — |
| Augmentation de données | ✅ swap / nlpaug | Marginal | — |
| Focal loss | ❌ | — | À tester |
| Réduire à 2 classes (positif/négatif) | ❌ | Potentiel +30 % F1 | À tester |
| Modèle pré-fine-tuné sentiment fr (tblard/tf-allocine) | ❌ | — | Baseline à comparer |

*La validation lexicale TF-IDF (CRITIQUE vs HOSTILITÉ = vocabulaires distincts) confirme que les classes sont bien discriminables. Le blocage vient du volume d'annotations pour le fine-tuning.*